# Genotype–Phenotype Heterogeneity in Non-Classical 21-Hydroxylase Deficiency: FAIR² Dataset Exploration with `mlcroissant`
This notebook guides you through loading, overview, and exploratory analysis of the FAIR² clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

FAIR² dataset contains structured tabulations of clinical, laboratory, imaging, and genetic data for three female patients with non-classical 21-hydroxylase deficiency-associated infertility. It is expressively documented and suited for case-based genotype–phenotype research.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n\nDescription: {metadata['description']}")
print(f"Published: {metadata.get('datePublished','N/A')}, Version: {metadata.get('version','N/A')}")

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id` identifiers.

Each table in the dataset is represented as a Croissant RecordSet, with columns defined by their unique `@id`.

Let's enumerate the RecordSets, their fields, and example records.

In [ ]:
# List all RecordSets by their @id
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name','(no name)')} - {rs.get('description','(no description)')}")

# For each RecordSet, enumerate fields (columns) by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name','(no name)')})")
    fields = rs.get('field',[])
    if not fields:
        print("  No fields defined.")
        continue
    for fld in fields:
        print(f"  Field @id: {fld['@id']} - Name: {fld.get('name','')} - Description: {fld.get('description','')}")

# Show a sample record for each RecordSet
for rs in record_sets:
    print(f"\nFirst record from {rs['@id']}:")
    recs = list(dataset.records(record_set=rs['@id']))
    if recs:
        print(recs[0])
    else:
        print("No records.")

## 3. Data Extraction
Load all records from each RecordSet into DataFrames for further analysis.

Use Croissant `@id` fields to refer to RecordSets and the columns.

In [ ]:
# Prepare a list of all RecordSet IDs
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load each RecordSet into a DataFrame using its @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for {record_set_id}:")
    print("Columns:", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filtering, normalization, and grouping.

We select Table 1 (clinical features) and analyze the 'Age at Diagnosis' (by its field `@id`) across patients, normalize values, and group by presence of infertility if available.

You can adapt this to other tables and fields using their Croissant IDs as shown.

In [ ]:
# Choose the clinical features table by its RecordSet @id
# Example RecordSet @id for Table 1:
clinical_rs_id = [rs['@id'] for rs in dataset.record_sets if 'Table 1' in rs.get('name','')][0] if dataset.record_sets else None
df = dataframes[clinical_rs_id]

# Find the numeric field '@id' for 'Age at Diagnosis'
clinical_fields = [f for f in dataset.get_record_set(clinical_rs_id).get('field',[]) if 'Age' in f.get('name','') or 'age' in f.get('name','')]
age_field = clinical_fields[0]['@id'] if clinical_fields else df.columns[0]

# Filter for 'age' above threshold and normalize
threshold = 25
filtered_df = df[df[age_field] > threshold]
print(f"Records with age > {threshold}:")
print(filtered_df[[age_field]].head())

# Normalize age
filtered_df[f"{age_field}_normalized"] = (filtered_df[age_field] - filtered_df[age_field].mean()) / filtered_df[age_field].std()
print(f"Normalized age:")
print(filtered_df[[age_field, f"{age_field}_normalized"]].head())

# Try grouping by 'Infertility' status, using @id
infertility_field = [f['@id'] for f in dataset.get_record_set(clinical_rs_id).get('field',[]) if 'Infertility' in f.get('name','') or 'infertility' in f.get('name','')]
group_field = infertility_field[0] if infertility_field else None
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[age_field].mean().to_frame()
    print(f"Grouped age by infertility status:")
    print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions or relationships (e.g., Age at Diagnosis vs. Infertility) with matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot age distribution
plt.figure(figsize=(7,4))
sns.histplot(df[age_field], bins=5, kde=True)
plt.title("Age at Diagnosis Distribution")
plt.xlabel("Age at Diagnosis")
plt.ylabel("Count")
plt.show()

# Plot age by infertility status if available
if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[group_field], y=df[age_field])
    plt.title("Age at Diagnosis by Infertility Status")
    plt.xlabel("Infertility")
    plt.ylabel("Age at Diagnosis")
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded FAIR² clinical dataset using the Croissant schema and `mlcroissant`.
- Reviewed structure via RecordSets and fields identified by their `@id`.
- Extracted tabular data per table and demonstrated basic EDA: filtering, normalization, grouping, and visualization.

This approach is extensible to additional tables within the dataset, and can be adapted for more advanced statistical and modeling workflows. Data entity referencing via `@id` ensures reproducible and FAIR analysis.

For more details, refer to the full Croissant schema and documentation at [mlcroissant](https://github.com/mlcommons/croissant).